In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
books = pd.read_csv("books_with_categories.csv")

books.columns

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)
classifier("I love this!")


In [ ]:
books["description"][0]

In [ ]:
classifier(books["description"][0])

In [ ]:
classifier(books["description"][0].split("."))

In [ ]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)

In [ ]:
sentences[6]

In [ ]:
predictions[6]

In [ ]:
sorted(predictions[0], key = lambda x: x['label'])

In [ ]:
emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
  per_emotion_scores = {label: [] for label in emotion_labels}
  for prediction in predictions:
    sorted_predictions = sorted(prediction, key=lambda x: x["label"])
    for index, label in enumerate(emotion_labels):
      per_emotion_scores[label].append(sorted_predictions[index]["score"])

  return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [ ]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
emotion_scores

In [ ]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(10)):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 10/10 [00:01<00:00,  8.74it/s]


In [ ]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn
emotions_df

,anger,disgust,fear,joy,sadness,neutral,isbn13
0,0.064134,0.273591,0.928168,0.932797,0.646216,0.967158,9780002005883
1,0.612619,0.348284,0.942528,0.704422,0.887939,0.111690,9780002261982
2,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,9780006178736
3,0.351484,0.150723,0.360706,0.251881,0.732685,0.111690,9780006280897
4,0.081412,0.184495,0.095043,0.040564,0.884390,0.475881,9780006280934
5,0.232225,0.727175,0.051363,0.043376,0.621393,0.111690,9780006380832
6,0.538184,0.155855,0.747429,0.872565,0.712194,0.408000,9780006470229
7,0.064134,0.104007,0.404496,0.040564,0.549477,0.820282,9780006472612
8,0.300670,0.279481,0.915524,0.040564,0.840290,0.354460,9780006482079
9,0.064134,0.177928,0.051363,0.040564,0.860372,0.111690,9780006483014


In [ ]:
books = pd.merge(books, emotions_df, on="isbn13")

In [ ]:
books

In [ ]:
books.to_csv("books_with_emotions.csv", index = False)
